In [2]:
# ====================================
# Import libraries
# ====================================
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

#NB. Multi Head Model
#Head 1 predicts the hour (12 classes: 0-11);
#Head 2 predicts the minutes (60 classes: 0-59);
#Classification for both tasks. 

#NB2. I will still try to reuse as much code as possible from the previous tasks

In [3]:
# ====================================
# Reproducibility and device
# ====================================
r_seed = 99
random.seed(r_seed)
np.random.seed(r_seed)
torch.manual_seed(r_seed)
generator = torch.Generator().manual_seed(r_seed)

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(f"Using device: {device}")


In [ ]:
# ====================================
# Load the Clock dataset (75x75)
# ====================================
images_path = "data/075_images.npy"
labels_path = "data/075_labels.npy"

images_np = np.load(images_path)
labels_np = np.load(labels_path)

#Check out your files
    #No longer needed at this point 

#Convert into pytorch tensors
images = torch.from_numpy(images_np).float() / 255.0 #turn numpy array into pytorch tensor, float, and rescale between 0 and 1
images = images.unsqueeze(1) #add channel dimension, we need format (batch_size, channels, height, width) (greyscale = 1 channel)

#Create two separate labels for minute and hour
labels_hour = torch.from_numpy(labels_np[:, 0]).long() #HOURS, pytorch tensor
labels_minute = torch.from_numpy(labels_np[:, 1]).long() #MINUTES, pytorch tensor
print(labels_minute.shape, labels_minute.shape) #18'000 labels for both

#We are trying to predict:
#Hours: from 0 to 11.
#Minutes: from 0 to 59.

torch.Size([18000]) torch.Size([18000])


In [6]:
#=====================================
# Train-Validation-Test split
# ====================================
dataset = TensorDataset(images, labels_hour, labels_minute) #Now we have 3 arguments
n = len(dataset)

train_size = int(0.8 * n)
valid_size = int(0.1 * n)
test_size = n - train_size - valid_size

train_data, valid_data, test_data = random_split(
    dataset,
    [train_size, valid_size, test_size],
    generator=generator
)

batch_size = 64

train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_data,
    batch_size=batch_size,
    shuffle=False 
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# =====================================
# Setting up the CNN
# =====================================

class Net(nn.Module):

    def __init__(self): #we try to use: 12 classes for hours, 60 classes for minutes
        super().__init__()
        #Input image (1, 75, 75) (greyscale, 75x75)
        
        #Same layers for both
        self.conv1 = nn.Conv2d(1, 6, 3) #convolutional layer 1: 6 filters, 3x3
        self.pool = nn.MaxPool2d(2, 2) #pooling to reduce dimension: 6 filters, 36x36
        self.conv2 = nn.Conv2d(6, 16, 3) #convolutional layer 2: 16 filters, 3x3 (input 6 filters 36x36)
        self.fc1 = nn.Linear(16 * 17 * 17, 120) #fully connected layer: receives 16 filters, pooled to 17x17
        self.fc2 = nn.Linear(120, 84)
        
        #Two separate outputs
        self.hour_head = nn.Linear(84, 12)
        self.minute_head = nn.Linear(84, 60)

    def forward(self, x):
        
        #Same forward for both
        x = self.pool(F.relu(self.conv1(x))) #Step 1: conv1 --> relu --> maxpool
        x = self.pool(F.relu(self.conv2(x))) #Step 2: conv2 --> relu --> maxpool
        x = torch.flatten(x, 1) #Step 3: flatten
        x = F.relu(self.fc1(x)) #Step 4: fully connected 1 --> relu
        x = F.relu(self.fc2(x)) #Step 5: fully connected 2 --> relu
        
        #Two separate outputs
        out_hour = self.hour_head(x) #Step 6: output
        out_minute = self.minute_head(x) 

        return out_hour, out_minute

In [ ]:
# ======================================================
# Defining the functions for training and evaluation
# ======================================================


#Train the model

def train_model(model, train_dataloader, optimizer, criterion, epochs, alpha): #alpha for combining the losses
    
    """Train a model for a given number of epochs."""
    
    for epoch in range(epochs):
        
        model.train() 
        running_loss = 0.0
        
        for inputs, hours, minutes in train_dataloader:

            #zero the parameter gradients
            optimizer.zero_grad()

            #forward + backward + optimize
            out_hour, out_minute = model(inputs) #predictions
            loss_hour = criterion(out_hour, hours) #Loss for HOURS
            loss_minute = criterion(out_minute, minutes) #Loss for MINUTES
            loss = loss_hour + alpha * loss_minute #COMBINE the Losses
            loss.backward() #compute gradients
            optimizer.step() #optimize weights

            #compute running loss
            running_loss += loss.item()
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"- Loss = {running_loss/len(train_dataloader):.4f}"
        )

#Validate the model

def evaluate_model(model, dataloader, bin_size):

    """Evaluate a model using the average circular time error."""

    model.eval()

    total_error = 0
    total = 0

    with torch.no_grad():
        for inputs, hours, minutes in dataloader:
            
            #Get predictions and classes
            out_hour, out_minute = model(inputs) #predictions 
            predicted_hours = torch.argmax(out_hour, dim=1) #Class for HOURS: assign to the class with the highest score
            predicted_minutes = torch.argmax(out_minute, dim=1) #Class for MINUTES: assign to the class with the highest score

            #Combine predictions and true times into total minutes
            predicted_time = predicted_hours * 60 + predicted_minutes #total number of minutes
            true_time = hours * 60 + minutes #total number of minutes
            
            #Calculate their absolute difference and their circular difference
            difference = torch.abs(predicted_time - true_time)
            circular_difference = torch.minimum(difference, 720 - difference)

            #Compute the errors
            total_error += circular_difference.sum().item()
            total += minutes.size(0)

    common_sense_error = total_error / total

    return common_sense_error

#NB. 
#While we use two separate Losses to train the model (and then combine them into one),
#We use one common sense error to check how off our predictions are.
#Therefore for the common sense error we compute again the total number of minutes, and 
#the circular difference, because it's convenient to estimate the error in our predictions.  

In [ ]:
# =====================================
# Experiment with different alpha values
# =====================================

learning_rate = 0.001
epochs = 20

#Alpha values: the value with which we combine the 
#two losses: L_tot = L_hour + alpha*L_minute
alpha_values = [0.5, 1.0, 2.0]

results = []

for alpha in alpha_values:

    print(f"\nTraining with alpha = {alpha}")

    #New model for each experiment
    model = Net()

    #Loss function
    criterion = nn.CrossEntropyLoss()
    
    #Optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    #Train the model
    train_model(
        model,
        train_dataloader,
        optimizer,
        criterion,
        epochs,
        alpha
    )

    #Validate the model
    validation_error = evaluate_model(
        model, 
        valid_dataloader
    )

    #Save results
    results.append({
        "Alpha": alpha,
        "Validation error (minutes)": validation_error
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Validation error (minutes)"
)

print(results_df)

In [ ]:
# =====================================
# Test set - With Best Parameters
# =====================================

#Best alpha
best_alpha = 1.0 

#New model
final_model = Net()

#Loss function
criterion = nn.CrossEntropyLoss()

#Optimizer
optimizer = optim.Adam(final_model.parameters(), lr=0.001)

#Train the final model on training set
train_model(
    final_model,
    train_dataloader,
    optimizer,
    criterion,
    epochs=20,
    alpha=best_alpha
)

#Evaluate the final model on test set
test_error = evaluate_model(
    final_model, 
    test_dataloader
)

print(f"Test error: {test_error:.2f} minutes")

In [ ]:
# ====================================
# Compare performances
# ====================================

#Compare the performance of:
# - Multi Head (separate min and hour)
# - Classification (together min and hour)
# - Regression (together min and hour)